# Export Excel des resultats de comparaison

Ce notebook relit les fichiers produits par `03e_tester_comparaison_branches.ipynb`, les affiche rapidement, puis cree un classeur Excel avec un onglet par fichier.


In [ ]:
from pathlib import Path
import sys
import pandas as pd


def find_project_root(start_path: Path) -> Path:
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src" / "compliance_nlp").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError("Impossible de trouver la racine du projet depuis le dossier courant.")


ROOT = find_project_root(Path.cwd().resolve())
EXPORT_DIR = ROOT / "outputs" / "comparaison_branches"
EXCEL_PATH = EXPORT_DIR / "comparaison_branches.xlsx"

pd.set_option("display.max_colwidth", 180)
{"export_dir": str(EXPORT_DIR), "excel_path": str(EXCEL_PATH)}


## Fichiers disponibles


In [ ]:
if not EXPORT_DIR.exists():
    raise FileNotFoundError(
        f"Dossier introuvable: {EXPORT_DIR}. Lancez d'abord le notebook 03e pour generer les CSV."
    )

csv_files = sorted(path for path in EXPORT_DIR.glob("*.csv") if path.name != "manifest.csv")
if not csv_files:
    raise FileNotFoundError(
        f"Aucun CSV trouve dans {EXPORT_DIR}. Lancez d'abord le notebook 03e jusqu'a la cellule d'export."
    )

files_df = pd.DataFrame([
    {
        "file": path.name,
        "size_bytes": path.stat().st_size,
        "modified_at": pd.Timestamp(path.stat().st_mtime, unit="s"),
        "path": str(path),
    }
    for path in csv_files
])
files_df


## Chargement et apercu


In [ ]:
tables = {path.stem: pd.read_csv(path, keep_default_na=False) for path in csv_files}

summary_df = pd.DataFrame([
    {
        "sheet": name[:31],
        "source_file": f"{name}.csv",
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
    }
    for name, dataframe in tables.items()
])
summary_df


In [ ]:
for name, dataframe in tables.items():
    print(f"\n=== {name} ({len(dataframe)} lignes, {len(dataframe.columns)} colonnes) ===")
    display(dataframe.head(10))


## Export Excel multi-onglets


In [ ]:
def safe_sheet_name(name: str, used_names: set[str]) -> str:
    cleaned = "".join(character if character not in "[]:*?/\\" else "_" for character in name)[:31] or "sheet"
    candidate = cleaned
    index = 1
    while candidate in used_names:
        suffix = f"_{index}"
        candidate = f"{cleaned[:31 - len(suffix)]}{suffix}"
        index += 1
    used_names.add(candidate)
    return candidate


EXPORT_DIR.mkdir(parents=True, exist_ok=True)
used_sheet_names = set()
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="sommaire", index=False)
    for name, dataframe in tables.items():
        sheet_name = safe_sheet_name(name, used_sheet_names)
        dataframe.to_excel(writer, sheet_name=sheet_name, index=False)

{"excel_path": str(EXCEL_PATH), "sheet_count": len(tables) + 1}
